# 01: 組み合わせ最適化問題のQUBOへのマッピング
### (Mapping Combinatorial Optimization Problems to QUBO)

---

## 1. Thesis: 科学的問いと仮説

量子アニーリングやQAOAなどの量子アルゴリズムを用いて現実世界の最適化問題を解くためには、まず問題を「量子コンピュータが理解できる形式」に変換する必要があります。その標準的な形式が **QUBO (Quadratic Unconstrained Binary Optimization)** です。

**問い** : 都市間のルート選択やグループ分けといった複雑な組み合わせ問題を、量子ビット間の「相互作用（重み）」という数学的形式（QUBO/Isingモデル）に変換する統一的な手法とはどのようなものか？

**仮説** : 目的関数を「コストが最小になる＝エネルギーが最小になる」ように2次形式の多項式で表現し、バイナリ変数 $x \in \{0, 1\}$ をスピン変数 $s \in \{-1, 1\}$ にマッピングすることで、あらゆるNP困難な組み合わせ問題をハミルトニアンの基底状態探索問題へと帰着させることができる。


## 2. Theoretical Background (理論的背景)

### 2.1 QUBOとは？
QUBOは、0または1をとるバイナリ変数 $x_i$ を用いた以下の2次形式の最小化問題です：
$$E(x) = \sum_{i} Q_{ii} x_i + \sum_{i<j} Q_{ij} x_i x_j$$
ここで、$Q_{ii}$ は各変数のバイアス幅（重み）、$Q_{ij}$ は変数間の相互作用の強さを表します。

### 2.2 Isingモデルへの変換
量子計算のハミルトニアンは通常、基底が $\{-1, 1\}$ の Ising モデル（スピンモデル）で表現されます。バイナリ変数 $x_i$ からスピン変数 $s_i$ への変換式は以下の通りです：
$$s_i = 1 - 2x_i \quad \text{または} \quad x_i = \frac{1 - s_i}{2}$$
この変換を施すことで、QUBOは行列形式のハミルトニアン $H = \sum h_i Z_i + \sum J_{ij} Z_i Z_j$ へと書き換えられ、量子ゲート（RZやZZ結合）による計算が可能になります。

### 2.3 Max-Cut問題の定式化
Max-Cut問題では、エッジ $(i, j)$ がカットされる（異なるグループに属する）ときにスコア $+1$ となります。これを最小化問題（エネルギー）にするには、各エッジに対し：
$$C_{ij}(x) = -(x_i(1-x_j) + x_j(1-x_i))$$
$$= x_i x_j + x_j x_i - x_i - x_j = 2x_i x_j - x_i - x_j$$
という項を合計します。


## 3. Implementation (実装)

隣接行列で定義されたグラフ構造から、QUBO行列 $Q$ および、Isingモデルの係数 $h, J$ を自動生成するプログラムを実装します。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def get_maxcut_qubo(num_nodes, edges):
    """グラフのエッジリストからQUBO行列を生成します。"""
    # QUBO行列 Q (N x N)
    Q = np.zeros((num_nodes, num_nodes))
    
    for i, j in edges:
        # Max-Cutのコスト項: -(xi + xj - 2xixj) = -xi - xj + 2xixj
        # ※ 最小化問題にするため負のカット数をエネルギーとする
        # 対角成分 (線形項)
        Q[i, i] += -1
        Q[j, j] += -1
        # 非対角成分 (相互作用項)
        Q[i, j] += 2
        
    return Q

def qubo_to_ising(Q):
    """QUBO行列からIsingモデルの係数 (h, J) に変換します。"""
    num_vars = len(Q)
    h = np.zeros(num_vars)
    J = {}
    
    # 定数項は無視
    for i in range(num_vars):
        # h_i = 1/2 * diagonal + 1/4 * sum of interactions
        h[i] += Q[i, i] / 2
        for j in range(i + 1, num_vars):
            interaction = Q[i, j]
            h[i] += interaction / 4
            h[j] += interaction / 4
            J[(i, j)] = interaction / 4
            
    return h, J

# 3ノードの直線グラフ (0--1, 1--2)
nodes = 3
edges = [(0, 1), (1, 2)]
Q_matrix = get_maxcut_qubo(nodes, edges)
h_vec, J_dict = qubo_to_ising(Q_matrix)

print("QUBO Matrix Q:")
print(Q_matrix)
print("\nIsing Coefficients:")
print(f"h (linear): {h_vec}")
print(f"J (quadratic): {J_dict}")


## 4. Visualization & Analysis (可視化と解析)

「どの変数が強く結びついているか」を可視化するQUBOヒートマップと、すべてのビットパターン（$2^N$ 通り）のエネルギーを表示するエネルギー景観図を作成します。


In [ ]:
# 1. QUBO行列のヒートマップ
plt.figure(figsize=(6, 5))
sns.heatmap(Q_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("QUBO Matrix Heatmap (Max-Cut)")
plt.show()

# 2. 全状態のエネルギー計算 (Brute-force)
energies = []
bitstrings = [format(i, f'0{nodes}b') for i in range(2**nodes)]

for b in bitstrings:
    x = np.array([int(bit) for bit in b])
    # E = x^T Q x
    energy = x.T @ Q_matrix @ x
    energies.append(energy)

# エネルギー景観のプロット
plt.figure(figsize=(10, 4))
plt.bar(bitstrings, energies, color=['#d62728' if e == min(energies) else '#1f77b4' for e in energies])
plt.axhline(0, color='black', lw=1)
plt.title("Energy Landscape (All Possible States)")
plt.ylabel("Energy (Cost)")
plt.xlabel("State |x2 x1 x0>")
plt.grid(axis='y', alpha=0.3)
plt.show()

min_val = min(energies)
opt_states = [bitstrings[i] for i, e in enumerate(energies) if e == min_val]
print(f"最小エネルギー: {min_val}")
print(f"最適解（基底状態）: {opt_states}")


## 5. Conclusion & Future Work (結論と展望)

### 結論
本ノートブックでは、抽象的な組み合わせ最適化問題を、量子計算の入力形式であるQUBOおよびIsingモデルへ変換するプロセスを確立しました。

1. **マッピングの普遍性** : グラフの構造がQUBO行列の「相互作用（非対角成分）」として正確にマッピングされることを確認しました。

2. **エネルギー関数の構築** : 全探索の結果、目的とするMax-Cutの解（`010` や `101`）が、定義したQUBO行列において最も低いエネルギー（コスト）を持つことが視覚的にも証明されました。

### 展望
- **制約条件の追加** : 現実の問題には「〜合計が10以下であること」といった制約が含まれます。これらをペナルティ項（ラグランジュ未定係数法）としてQUBOに組み込む手法の学習が次のステップとなります。

- **スケーラビリティ** : ノード数が増えた際、QUBO行列は $N^2$ のサイズで増大します。疎行列形式での保持や、ハードウェアの接続トポロジー（キメラグラフやペガサスグラフ）への埋め込み技術が実用化における鍵となります。
